# Qwen72B Colab Generation Notebook

This notebook runs the current repository's Phase 4 data-generation flow entirely inside one Colab H100 session.

It uses the live repo scripts instead of the older Phase 7 reference CLI:

- `scripts/ingest_handbooks.py` builds `data/raw_chunks.jsonl`.
- `scripts/generate_qa.py` connects to a local vLLM OpenAI-compatible endpoint and writes `data/qa_pairs.jsonl`.

Recommended workflow:

1. Run the cells from top to bottom.
2. Leave `ACTIVE_PROFILE = "qwen72b_gptq"` first. If startup fails, switch to `"qwen32b"`.
3. Leave `TEST_MODE = False` for the full run, or switch it to `True` for a quick 10-chunk smoke test.
4. Download the generated files from `/content/TotNghiepProject/data/` or the zip created near the end.

Important runtime note: this notebook forces the safe vLLM sampler path by setting `VLLM_USE_FLASHINFER_SAMPLER=0`. That avoids the FlashInfer startup crash seen on some Colab runtimes, even when the GPU itself is valid.

This notebook does not require ngrok because generation runs inside Colab against `http://127.0.0.1:8000`.

In [1]:
%%capture
# Install vLLM onto a clean torch stack so Colab does not keep an incompatible default wheel.
%pip uninstall -y torch torchvision torchaudio xformers 2>/dev/null || true
%pip install -q "vllm>=0.6.0" "openai>=1.30.0" "httpx>=0.27.0" "langchain-text-splitters>=0.3.0"

In [2]:
import json
import os
import secrets
import shutil
import subprocess
import sys
import time
from pathlib import Path
from urllib import request as urllib_request

import torch

try:
    from google.colab import userdata
except Exception:
    userdata = None

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["TQDM_DISABLE"] = "1"

REPO_URL = "https://github.com/ndn2k5/TotNghiepProject"
REPO_BRANCH = "main"
WORKDIR = Path("/content/qwen72b_colab")
REPO_ROOT = Path("/content/TotNghiepProject")
CACHE_DIR = WORKDIR / "hf_cache"
LOG_DIR = WORKDIR / "logs"
SERVER_LOG = LOG_DIR / "vllm_server.log"

PORT = 8000
API_BASE = f"http://127.0.0.1:{PORT}"
TEST_MODE = False  # set True if you want a 10-chunk smoke test first
REBUILD_RAW_CHUNKS = False  # set True to force a fresh rebuild of data/raw_chunks.jsonl
RESET_QA_OUTPUTS = False  # set True only when you want to delete qa_pairs/log/checkpoint and restart from zero
UPDATE_REPO_IF_PRESENT = False  # set True if you want the notebook to pull the latest main branch
USE_FLASHINFER_SAMPLER = False  # keep the safe sampler path; avoids Colab FlashInfer startup failures
ACTIVE_PROFILE = "qwen72b_gptq"

MODEL_PROFILES = {
    "qwen72b_gptq": {
        "model_id": "Qwen/Qwen2.5-72B-Instruct-GPTQ-Int8",
        "gpu_memory_utilization": 0.96,
        "max_model_len": 4096,
        "max_num_seqs": 4,
        "reason": "Best-quality single-H100 profile. Use this first.",
    },
    "qwen32b": {
        "model_id": "Qwen/Qwen2.5-32B-Instruct",
        "gpu_memory_utilization": 0.94,
        "max_model_len": 4096,
        "max_num_seqs": 6,
        "reason": "Fallback profile if 72B startup is too tight or too slow.",
    },
}

if ACTIVE_PROFILE not in MODEL_PROFILES:
    raise ValueError(f"Unsupported ACTIVE_PROFILE: {ACTIVE_PROFILE}")

PROFILE = MODEL_PROFILES[ACTIVE_PROFILE]

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN and userdata is not None:
    try:
        HF_TOKEN = (userdata.get("HF_TOKEN") or "").strip()
    except Exception:
        HF_TOKEN = ""

WORKDIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR)
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "1" if USE_FLASHINFER_SAMPLER else "0"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
).stdout.strip()

gpu_name = None
gpu_capability = None
gpu_capability_tuple = None
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capability_tuple = torch.cuda.get_device_capability(0)
    major, minor = gpu_capability_tuple
    gpu_capability = f"sm{major}{minor} ({major}.{minor})"

if gpu_capability_tuple is not None and gpu_capability_tuple < (7, 5):
    raise RuntimeError(
        f"Current Colab GPU {gpu_name} reports compute capability {gpu_capability}, below sm75. "
        "Reconnect until you get T4/L4/A100/H100-class hardware. vLLM will not be reliable on this runtime."
    )

print(json.dumps({
    "repo_url": REPO_URL,
    "repo_branch": REPO_BRANCH,
    "active_profile": ACTIVE_PROFILE,
    "model_id": PROFILE["model_id"],
    "gpu_memory_utilization": PROFILE["gpu_memory_utilization"],
    "max_model_len": PROFILE["max_model_len"],
    "max_num_seqs": PROFILE["max_num_seqs"],
    "test_mode": TEST_MODE,
    "rebuild_raw_chunks": REBUILD_RAW_CHUNKS,
    "reset_qa_outputs": RESET_QA_OUTPUTS,
    "update_repo_if_present": UPDATE_REPO_IF_PRESENT,
    "flashinfer_sampler_enabled": USE_FLASHINFER_SAMPLER,
    "cache_dir": str(CACHE_DIR),
    "server_log": str(SERVER_LOG),
    "gpu_capability": gpu_capability,
}, indent=2))
print("GPU:", gpu_query or "unavailable")

{
  "repo_url": "https://github.com/ndn2k5/TotNghiepProject",
  "repo_branch": "main",
  "active_profile": "qwen72b_gptq",
  "model_id": "Qwen/Qwen2.5-72B-Instruct-GPTQ-Int8",
  "gpu_memory_utilization": 0.96,
  "max_model_len": 4096,
  "max_num_seqs": 4,
  "test_mode": false,
  "rebuild_raw_chunks": false,
  "reset_qa_outputs": false,
  "update_repo_if_present": false,
  "flashinfer_sampler_enabled": false,
  "cache_dir": "/content/qwen72b_colab/hf_cache",
  "server_log": "/content/qwen72b_colab/logs/vllm_server.log",
  "gpu_capability": "sm120 (12.0)"
}
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB


In [3]:
def tail_log(path: Path, lines: int = 60) -> None:
    if not path.exists():
        print(f"{path} does not exist yet.")
        return
    content = path.read_text(encoding="utf-8", errors="ignore").splitlines()
    start = max(len(content) - lines, 0)
    for line in content[start:]:
        print(line)

def run_checked(cmd: list[str], cwd: Path | None = None) -> subprocess.CompletedProcess:
    print("$", " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
        check=False,
    )
    if result.stdout.strip():
        print(result.stdout[-4000:])
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr[-4000:])
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result

def ensure_repo() -> Path:
    if REPO_ROOT.exists():
        print(f"Reusing existing repo at {REPO_ROOT}")
        if UPDATE_REPO_IF_PRESENT:
            run_checked(["git", "pull", "--ff-only"], cwd=REPO_ROOT)
        return REPO_ROOT
    run_checked([
        "git",
        "clone",
        "--branch",
        REPO_BRANCH,
        "--depth",
        "1",
        REPO_URL,
        str(REPO_ROOT),
    ])
    return REPO_ROOT

def ensure_raw_chunks() -> Path:
    raw_chunk_path = REPO_ROOT / "data" / "raw_chunks.jsonl"
    if REBUILD_RAW_CHUNKS and raw_chunk_path.exists():
        raw_chunk_path.unlink()
    if raw_chunk_path.exists():
        line_count = sum(1 for _ in raw_chunk_path.open(encoding="utf-8"))
        print(f"Using existing raw chunks: {raw_chunk_path} ({line_count} rows)")
        return raw_chunk_path
    run_checked([sys.executable, "scripts/ingest_handbooks.py"], cwd=REPO_ROOT)
    line_count = sum(1 for _ in raw_chunk_path.open(encoding="utf-8"))
    print(f"Built raw chunks: {raw_chunk_path} ({line_count} rows)")
    return raw_chunk_path

def prepare_generation_outputs() -> None:
    for relative_path in [
        "data/qa_pairs.jsonl",
        "data/qa_generation_log.txt",
        "data/qa_checkpoint.json",
    ]:
        target = REPO_ROOT / relative_path
        if RESET_QA_OUTPUTS and target.exists():
            target.unlink()
            print(f"Removed {target}")

def stop_vllm_server() -> None:
    subprocess.run("pkill -f 'vllm.entrypoints.openai.api_server'", shell=True, check=False)

def start_vllm_server() -> subprocess.Popen:
    stop_vllm_server()
    if SERVER_LOG.exists():
        SERVER_LOG.unlink()
    cmd = [
        sys.executable,
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        PROFILE["model_id"],
        "--host",
        "127.0.0.1",
        "--port",
        str(PORT),
        "--gpu-memory-utilization",
        str(PROFILE["gpu_memory_utilization"]),
        "--max-model-len",
        str(PROFILE["max_model_len"]),
        "--max-num-seqs",
        str(PROFILE["max_num_seqs"]),
        "--trust-remote-code",
        "--disable-log-stats",
    ]
    with SERVER_LOG.open("w", encoding="utf-8") as handle:
        process = subprocess.Popen(
            cmd,
            stdout=handle,
            stderr=subprocess.STDOUT,
            cwd=str(WORKDIR),
            env=os.environ.copy(),
        )
    return process

def wait_for_vllm_ready(
    server_process: subprocess.Popen | None = None,
    timeout_minutes: int | None = None,
    status_interval_seconds: int = 60,
    log_tail_interval_seconds: int = 300,
) -> dict:
    deadline = None if timeout_minutes is None else time.time() + timeout_minutes * 60
    last_error = None
    last_status_at = 0.0
    last_log_tail_at = 0.0
    while True:
        if server_process is not None:
            exit_code = server_process.poll()
            if exit_code is not None:
                print(f"vLLM exited early with code {exit_code}. Last 120 server log lines:")
                tail_log(SERVER_LOG, lines=120)
                raise RuntimeError(
                    "vLLM exited before becoming ready. The model likely failed during download, load, or KV-cache allocation."
                )
        try:
            with urllib_request.urlopen(f"{API_BASE}/v1/models", timeout=20) as response:
                payload = json.loads(response.read().decode("utf-8"))
                return payload
        except Exception as error:
            last_error = error
            now = time.time()
            if now - last_status_at >= status_interval_seconds:
                wait_mode = "without a timeout" if deadline is None else f"until timeout ({timeout_minutes} minutes)"
                print(f"Still waiting for vLLM {wait_mode}. Last readiness error: {type(error).__name__}: {error}")
                last_status_at = now
            if now - last_log_tail_at >= log_tail_interval_seconds:
                print("Latest server log tail while waiting:")
                tail_log(SERVER_LOG, lines=40)
                last_log_tail_at = now
            if deadline is not None and now >= deadline:
                print("Timed out waiting for vLLM. Last 120 server log lines:")
                tail_log(SERVER_LOG, lines=120)
                raise RuntimeError(f"vLLM did not become ready in time: {last_error}")
            time.sleep(5)

def run_generation() -> None:
    cmd = [
        sys.executable,
        "scripts/generate_qa.py",
        "--vllm-url",
        API_BASE,
        "--model",
        PROFILE["model_id"],
    ]
    if TEST_MODE:
        cmd.append("--test")
    print("$", " ".join(cmd))
    process = subprocess.Popen(
        cmd,
        cwd=str(REPO_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line.rstrip())
    exit_code = process.wait()
    if exit_code != 0:
        raise RuntimeError(f"Generation failed with exit code {exit_code}")

def zip_outputs() -> Path:
    archive_base = Path(f"/content/qa_generation_{ACTIVE_PROFILE}")
    archive_path = archive_base.with_suffix(".zip")
    if archive_path.exists():
        archive_path.unlink()
    created_path = shutil.make_archive(
        base_name=str(archive_base),
        format="zip",
        root_dir=str(REPO_ROOT / "data"),
        base_dir=".",
    )
    return Path(created_path)

In [4]:
repo_root = ensure_repo()
raw_chunk_path = ensure_raw_chunks()

sample_rows = []
with raw_chunk_path.open(encoding="utf-8") as handle:
    for _, line in zip(range(2), handle):
        sample_rows.append(json.loads(line))

print(f"Repo root: {repo_root}")
print(f"Raw chunk file: {raw_chunk_path}")
print(f"Sample chunk count preview: {len(sample_rows)} rows loaded for inspection")
for index, row in enumerate(sample_rows, start=1):
    print()
    print(f"Chunk sample {index}: {row['source']} / {row['filename']}")
    print(row['text'][:500])

$ git clone --branch main --depth 1 https://github.com/ndn2k5/TotNghiepProject /content/TotNghiepProject
Using existing raw chunks: /content/TotNghiepProject/data/raw_chunks.jsonl (1503 rows)
Repo root: /content/TotNghiepProject
Raw chunk file: /content/TotNghiepProject/data/raw_chunks.jsonl
Sample chunk count preview: 2 rows loaded for inspection

Chunk sample 1: hshadab / Clef Values.md
# Clef Core Values

## Be better today than yesterday.

Being better is more important than being the "best", because "best" means complacency and better means progress. We are always improving. Improvement requires reflection, so we take ownership of our mistakes and question our habits.

## Treat others the way they'd like to be treated.

Chunk sample 2: hshadab / Clef Values.md
Our customers and teammates have different needs from our own, so we must consider their  perspectives to communicate effectively. We work best together when we empathize with one another, and we create the best product when

In [5]:
server_process = start_vllm_server()
print(f"Started vLLM with PID {server_process.pid}")
print(f"Model: {PROFILE['model_id']}")
print(f"Server log: {SERVER_LOG}")
print("Short initial server log tail:")
time.sleep(3)
tail_log(SERVER_LOG, lines=25)

Started vLLM with PID 3209
Model: Qwen/Qwen2.5-72B-Instruct-GPTQ-Int8
Server log: /content/qwen72b_colab/logs/vllm_server.log
Short initial server log tail:


In [6]:
timeout_minutes = None  # set an integer only if you want a hard deadline
print("Waiting for vLLM readiness. Status updates will print while the model is loading.")
models_payload = wait_for_vllm_ready(server_process=server_process, timeout_minutes=timeout_minutes)
print("vLLM is ready.")
print(json.dumps(models_payload, indent=2))
print("Latest server log tail:")
tail_log(SERVER_LOG, lines=40)

Waiting for vLLM readiness. Status updates will print while the model is loading.
Still waiting for vLLM without a timeout. Last readiness error: URLError: <urlopen error [Errno 111] Connection refused>
Latest server log tail while waiting:
Still waiting for vLLM without a timeout. Last readiness error: URLError: <urlopen error [Errno 111] Connection refused>
Still waiting for vLLM without a timeout. Last readiness error: URLError: <urlopen error [Errno 111] Connection refused>
Still waiting for vLLM without a timeout. Last readiness error: URLError: <urlopen error [Errno 111] Connection refused>
Still waiting for vLLM without a timeout. Last readiness error: URLError: <urlopen error [Errno 111] Connection refused>
Still waiting for vLLM without a timeout. Last readiness error: URLError: <urlopen error [Errno 111] Connection refused>
Latest server log tail while waiting:
Loading safetensors checkpoint shards:  30% Completed | 6/20 [00:01<00:04,  2.94it/s]
(EngineCore pid=3804) 
Loading

In [7]:
prepare_generation_outputs()
run_generation()

$ /usr/bin/python3 scripts/generate_qa.py --vllm-url http://127.0.0.1:8000 --model Qwen/Qwen2.5-72B-Instruct-GPTQ-Int8


KeyboardInterrupt: 

In [9]:
import csv
import json
from pathlib import Path

qa_path = Path("/content/TotNghiepProject/data/qa_pairs.jsonl")
csv_path = Path("/content/TotNghiepProject/data/qa_training_data.csv")

rows = []
with qa_path.open(encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)
        rows.append({
            "question": row["question"],
            "answer": row["answer"],
        })

with csv_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["question", "answer"])
    writer.writeheader()
    writer.writerows(rows)

print({"csv_path": str(csv_path), "rows": len(rows)})

{'csv_path': '/content/TotNghiepProject/data/qa_training_data.csv', 'rows': 730}


In [11]:
import csv
import json
from pathlib import Path

source_path = Path("/content/TotNghiepProject/data/qa_pairs.jsonl")  # change to .json if needed
csv_path = Path("/content/TotNghiepProject/data/qa_training_data.csv")

rows = []
with source_path.open(encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))

with csv_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["question", "answer"])
    writer.writeheader()
    for row in rows:
        writer.writerow({
            "question": row["question"],
            "answer": row["answer"],
        })

print({"csv_path": str(csv_path), "rows": len(rows)})

{'csv_path': '/content/TotNghiepProject/data/qa_training_data.csv', 'rows': 730}


In [ ]:
qa_path = REPO_ROOT / "data" / "qa_pairs.jsonl"
checkpoint_path = REPO_ROOT / "data" / "qa_checkpoint.json"
log_path = REPO_ROOT / "data" / "qa_generation_log.txt"

qa_rows = sum(1 for _ in qa_path.open(encoding="utf-8")) if qa_path.exists() else 0
checkpoint_payload = json.loads(checkpoint_path.read_text(encoding="utf-8")) if checkpoint_path.exists() else {}
processed_chunks = len(checkpoint_payload.get("processed", [])) if isinstance(checkpoint_payload, dict) else 0

print(json.dumps({
    "qa_pairs_path": str(qa_path),
    "qa_pairs_rows": qa_rows,
    "checkpoint_path": str(checkpoint_path),
    "processed_chunks": processed_chunks,
    "log_path": str(log_path),
}, indent=2))

if qa_path.exists():
    print()
    print("First 2 generated rows:")
    with qa_path.open(encoding="utf-8") as handle:
        for _, line in zip(range(2), handle):
            print(json.dumps(json.loads(line), ensure_ascii=False, indent=2))
            print()

archive_path = zip_outputs()
print(f"Created archive: {archive_path}")
print("Download the zip from the Colab Files panel if you want the whole data directory.")

In [10]:
stop_vllm_server()
print("Stopped the local vLLM server.")

Stopped the local vLLM server.


In [12]:
import shutil
from pathlib import Path
from google.colab import files

data_dir = Path("/content/TotNghiepProject/data")
archive_stem = "/content/totnghiepproject-data"
archive_path = Path(f"{archive_stem}.zip")

if archive_path.exists():
    archive_path.unlink()

shutil.make_archive(
    base_name=archive_stem,
    format="zip",
    root_dir=str(data_dir.parent),
    base_dir=data_dir.name,
)

print(f"Created archive: {archive_path}")
files.download(str(archive_path))

Created archive: /content/totnghiepproject-data.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>